Необходимо спарсить какой-либо сайт с вакансиями и по имеющейся там информации научиться восстанавливать ЗП, если она не указана

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
import time
import random

%matplotlib inline

In [ ]:
from fake_useragent import UserAgent

ua = UserAgent()
random_ua = ua.random

In [ ]:
url_habr = "https://career.habr.com/vacancies?page={num}&s[]=2&s[]=3&s[]=4&s[]=82&s[]=72&s[]=5&s[]=75&s[]=6&s[]=1&s[]=77&s[]=7&s[]=83&s[]=84&s[]=73&s[]=8&s[]=85&s[]=86&s[]=188&s[]=178&s[]=106&s[]=12&s[]=10&s[]=13&s[]=87&s[]=11&s[]=14&s[]=15&s[]=16&s[]=107&s[]=98&s[]=41&s[]=42&s[]=43&s[]=168&s[]=99&s[]=76&s[]=96&s[]=97&s[]=95&s[]=100&s[]=133&s[]=111&s[]=78&s[]=21&s[]=172&s[]=174&s[]=79&s[]=173&s[]=80&s[]=176&s[]=81&s[]=118&s[]=182&s[]=44&s[]=125&s[]=177&s[]=175&s[]=126&s[]=49&s[]=45&s[]=46&s[]=50&s[]=47&s[]=48&s[]=101&s[]=112&s[]=94&s[]=23&s[]=24&s[]=30&s[]=25&s[]=27&s[]=26&s[]=90&s[]=28&s[]=91&s[]=92&s[]=29&s[]=93&s[]=122&s[]=127&s[]=128&s[]=32&s[]=33&s[]=34&s[]=119&s[]=37&s[]=171&s[]=74&s[]=22&s[]=17&s[]=18&s[]=183&s[]=19&s[]=187&s[]=20&s[]=89&s[]=129&s[]=130&s[]=35&s[]=70&s[]=9&s[]=66&type=all&with_salary=true"

In [ ]:
with open('vacancies_urls.txt', mode='w') as file:
  c = 0

  for i in range(1, 6):
    response = requests.get(url_habr.format(num=i), headers={'User-Agent': random_ua})

    if response.status_code == 200:
      soup = BeautifulSoup(response.content)

      cards = soup.find_all('a', class_='vacancy-card__backdrop-link')
      print(len(cards))

      for card in cards:

        vacancy_id = card.get('href').split("/")[-1]
        file.write(vacancy_id + "\n")

        c += 1
        print(f"Записано новое значение: {vacancy_id}, #{c}")



25
Записано новое значение: 1000167783, #1
Записано новое значение: 1000166280, #2
Записано новое значение: 1000168633, #3
Записано новое значение: 1000167600, #4
Записано новое значение: 1000168182, #5
Записано новое значение: 1000168181, #6
Записано новое значение: 1000167775, #7
Записано новое значение: 1000168179, #8
Записано новое значение: 1000168756, #9
Записано новое значение: 1000166519, #10
Записано новое значение: 1000166713, #11
Записано новое значение: 1000166710, #12
Записано новое значение: 1000166709, #13
Записано новое значение: 1000167841, #14
Записано новое значение: 1000167763, #15
Записано новое значение: 1000167762, #16
Записано новое значение: 1000168168, #17
Записано новое значение: 1000168533, #18
Записано новое значение: 1000168162, #19
Записано новое значение: 1000168828, #20
Записано новое значение: 1000168091, #21
Записано новое значение: 1000163346, #22
Записано новое значение: 1000168291, #23
Записано новое значение: 1000167255, #24
Записано новое значени

In [ ]:
habr_vacancy_url = 'https://career.habr.com/vacancies/{vacancy_id}'

In [ ]:
df = pd.DataFrame(data={
    "id": [],
    "title": [],
    "salary": [],
    "requirements": [],
    "conditions": [],
    "company": [],
    "desc": [],
    "published_at": [],
    "link": [],
    "source": []
})

In [ ]:
df

,id,title,salary,requirements,conditions,company,desc,published_at,link,source


In [ ]:
with open('vacancies_urls.txt', mode='r') as file:
  vacancies = file.readlines()

  for vacancy_id in vacancies:
    print(f"Обработка вакансии {vacancy_id}")
    url = habr_vacancy_url.format(vacancy_id=vacancy_id.strip())
    #url = habr_vacancy_url.format(vacancy_id=1000167858)
    try:

      response = requests.get(url, headers={'User-Agent': random_ua})

      if response.status_code == 200:
        soup = BeautifulSoup(response.content)


        title = soup.find('h1', class_="page-title__title").text
        print(1)

        salary = soup.find('div', class_="basic-salary")
        if salary is None:
          print(f"Вакансия {vacancy_id} без зарплаты")

        salary = salary.text

        requirements = []
        vac_meta = soup.find_all('div', class_="vacancy-meta")[0]

        req1 = vac_meta.find_all('div', class_='basic-chip basic-chip--color-ui-turquoise basic-chip basic-chip--color-ui-turquoise')
        for req in req1:
          requirements.append(req.find('div', class_="chip-with-icon__text").text)

        req2 = vac_meta.find_all('div', class_='basic-chip basic-chip--color-ui-gray-4 basic-chip basic-chip--color-ui-gray-4')
        for req in req2:
          requirements.append(req.find('div', class_="chip-without-icon__text").text)

        requirements = ", ".join(requirements)
        print(3)

        conditions = []

        vac_meta2 = soup.find_all('div', class_="vacancy-meta")[1]
        conds = vac_meta2.find_all('div', class_='basic-chip basic-chip--color-ui-gray-4 basic-chip basic-chip--color-ui-gray-4')

        for cond in conds:
          conditions.append(cond.find('div', class_='chip-with-icon__text').text)

        conditions = ", ".join(conditions)
        print(4)

        company = soup.find('a', class_='link-comp link-comp--appearance-dark').text
        print(5)

        desc = soup.find('div', class_='vacancy-description__text').text
        print(6)

        published_at = soup.find('time', class_='basic-date').get('datetime')
        print(7)


        df.loc[len(df)] = {
              "id": vacancy_id,
              "title": title,
              "salary": salary,
              "requirements": requirements,
              "conditions": conditions,
              "company": company,
              "desc": desc,
              "published_at": published_at,
              "link": url,
              "source": 'habr'
      }
      print(f"Загружена вакансия {vacancy_id}")

    except Exception as e:
      df.to_csv("habr.csv", index=False)
      print(f"Произошла ошибка на странице {url}")
      print(f"{e}")

    time.sleep(random.randint(3, 15))

Обработка вакансии 1000167783

1
3
4
5
6
7
Загружена вакансия 1000167783

Обработка вакансии 1000166280

1
3
4
5
6
7
Загружена вакансия 1000166280

Обработка вакансии 1000168633

1
3
4
5
6
7
Загружена вакансия 1000168633

Обработка вакансии 1000167600

1
3
4
5
6
7
Загружена вакансия 1000167600

Обработка вакансии 1000168182

1
3
4
5
6
7
Загружена вакансия 1000168182

Обработка вакансии 1000168181

1
3
4
5
6
7
Загружена вакансия 1000168181

Обработка вакансии 1000167775

1
3
4
5
6
7
Загружена вакансия 1000167775

Обработка вакансии 1000168179

1
3
4
5
6
7
Загружена вакансия 1000168179

Обработка вакансии 1000168756

1
3
4
5
6
7
Загружена вакансия 1000168756

Обработка вакансии 1000166519

1
3
4
5
6
7
Загружена вакансия 1000166519

Обработка вакансии 1000166713

1
3
4
5
6
7
Загружена вакансия 1000166713

Обработка вакансии 1000166710

1
3
4
5
6
7
Загружена вакансия 1000166710

Обработка вакансии 1000166709

1
3
4
5
6
7
Загружена вакансия 1000166709

Обработка вакансии 1000167841

1
3
4
5

In [ ]:
df.to_excel("habr.xlsx", index=False)
